In [ ]:
import pandas as pd

# Input and output files
input_tsv = "aor0-5_merged.tsv"
output_tsv = "aor0-5_SM00066.tsv"

# Read without a header
df = pd.read_csv(input_tsv, sep="\t", header=None)

# Test whether any column in each row contains "SM00066"
mask = df.apply(
    lambda row: row.astype(str).str.contains("SM00066").any(),
    axis=1
)

# Extract
df_filtered = df[mask]

# Save as a TSV file without a header
df_filtered.to_csv(output_tsv, sep="\t", index=False, header=False)

print(f"Extraction complete: {output_tsv} ({len(df_filtered)} rows)")


In [ ]:
from pathlib import Path

TSV_PATH = "aor0-5_SM00066.tsv"   # TSV without a header
FAA_PATH = "aor0-5_2.faa"         # amino acid sequences of all genes
OUT_FASTA = "aor0-5_SM00066_domains.faa"


def read_fasta_as_dict(faa_file):
    seqs = {}
    cur_id = None
    cur_seq = []

    with open(faa_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    seqs[cur_id] = "".join(cur_seq)
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line.strip())

        if cur_id is not None:
            seqs[cur_id] = "".join(cur_seq)

    return seqs


def main():
    for p in (TSV_PATH, FAA_PATH):
        if not Path(p).exists():
            raise FileNotFoundError(f"File not found: {p}")

    seq_dict = read_fasta_as_dict(FAA_PATH)

    n_written = 0
    n_missing_id = 0
    n_bad_coord = 0

    with open(TSV_PATH, "r", encoding="utf-8") as tsv, \
         open(OUT_FASTA, "w", encoding="utf-8") as out:

        for line_no, line in enumerate(tsv, start=1):
            line = line.rstrip("\n")
            if not line:
                continue

            cols = line.split("\t")

            try:
                gene_id = cols[0]
                start = int(cols[6])
                end = int(cols[7])
            except (IndexError, ValueError):
                n_bad_coord += 1
                continue

            if gene_id not in seq_dict:
                n_missing_id += 1
                continue

            seq = seq_dict[gene_id]
            seqlen = len(seq)

            # 1-based inclusive coordinate check
            if start < 1 or end < 1 or start > end or end > seqlen:
                n_bad_coord += 1
                continue

            # Extract the sequence including the start and end amino acids
            subseq = seq[start - 1 : end]

            header = f">{gene_id}|SM00066|{start}-{end}|len={len(subseq)}|src_line={line_no}"
            out.write(header + "\n")
            out.write(subseq + "\n")

            n_written += 1

    print(f"Done.")
    print(f"Written sequences: {n_written}")
    print(f"Missing IDs in FASTA: {n_missing_id}")
    print(f"Bad/invalid coordinates: {n_bad_coord}")
    print(f"Output file: {OUT_FASTA}")


if __name__ == "__main__":
    main()


In [ ]:
from collections import defaultdict
from pathlib import Path

TSV_PATH = "aor0-5_SM00066.tsv"      # TSV without a header
FAA_PATH = "aor0-5.faa"              # amino acid sequences of all genes
OUT_FASTA = "aor0-5_non_SM00066_domains.faa"


def read_fasta_as_dict(faa_file):
    seqs = {}
    cur_id = None
    cur_seq = []

    with open(faa_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if cur_id is not None:
                    seqs[cur_id] = "".join(cur_seq)

                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)

        if cur_id is not None:
            seqs[cur_id] = "".join(cur_seq)

    return seqs


def main():
    for path in (TSV_PATH, FAA_PATH):
        if not Path(path).exists():
            raise FileNotFoundError(f"File not found: {path}")

    seq_dict = read_fasta_as_dict(FAA_PATH)

    # Store TSV records by ID
    records_by_id = defaultdict(list)
    bad_lines = []

    with open(TSV_PATH, "r", encoding="utf-8") as tsv:
        for line_no, line in enumerate(tsv, start=1):
            line = line.rstrip("\n")

            if not line:
                continue

            cols = line.split("\t")

            try:
                gene_id = cols[0]
                protein_length = int(cols[2])   # column 3
                domain_end = int(cols[7])       # column 8
            except (IndexError, ValueError):
                bad_lines.append(line_no)
                continue

            records_by_id[gene_id].append(
                {
                    "protein_length": protein_length,
                    "domain_end": domain_end,
                    "line_no": line_no,
                }
            )

    # If an ID occurs multiple times, use the record with the largest value in column 8
    selected_records = {}
    duplicate_ids = []

    for gene_id, records in records_by_id.items():
        if len(records) > 1:
            duplicate_ids.append(gene_id)

        selected_records[gene_id] = max(
            records,
            key=lambda record: record["domain_end"]
        )

    skipped_half_ids = []
    missing_ids = []
    invalid_coordinate_ids = []
    length_mismatch_ids = []
    n_written = 0

    with open(OUT_FASTA, "w", encoding="utf-8") as out:
        for gene_id, record in selected_records.items():
            protein_length = record["protein_length"]
            domain_end = record["domain_end"]
            line_no = record["line_no"]

            # Do not output the record if column 8 is at least half of column 3
            if domain_end >= protein_length / 2:
                skipped_half_ids.append(gene_id)
                continue

            if gene_id not in seq_dict:
                missing_ids.append(gene_id)
                continue

            seq = seq_dict[gene_id]
            fasta_length = len(seq)

            if fasta_length != protein_length:
                length_mismatch_ids.append(
                    (gene_id, protein_length, fasta_length)
                )

            # start = column 8 + 1、end = column 3
            start = domain_end + 1
            end = protein_length

            # Validate 1-based inclusive coordinates
            if start < 1 or end < 1 or start > end or end > fasta_length:
                invalid_coordinate_ids.append(gene_id)
                continue

            subseq = seq[start - 1:end]

            header = (
                f">{gene_id}|non_SM00066|{start}-{end}"
                f"|len={len(subseq)}|src_line={line_no}"
            )
            out.write(header + "\n")
            out.write(subseq + "\n")

            n_written += 1

    print("Done.")
    print(f"Written sequences: {n_written}")
    print(f"Output file: {OUT_FASTA}")

    print("\n[Duplicate IDs: largest 8th-column value selected]")
    if duplicate_ids:
        for gene_id in sorted(duplicate_ids):
            values = sorted(
                record["domain_end"]
                for record in records_by_id[gene_id]
            )
            selected_value = selected_records[gene_id]["domain_end"]
            print(
                f"{gene_id}\t8th-column values={values}"
                f"\tselected={selected_value}"
            )
    else:
        print("None")

    print("\n[Excluded IDs: 8th-column value >= half of 3rd-column value]")
    if skipped_half_ids:
        for gene_id in sorted(skipped_half_ids):
            record = selected_records[gene_id]
            print(
                f"{gene_id}\t3rd={record['protein_length']}"
                f"\t8th={record['domain_end']}"
            )
    else:
        print("None")

    print("\n[IDs missing from FASTA]")
    if missing_ids:
        for gene_id in sorted(missing_ids):
            print(gene_id)
    else:
        print("None")

    print("\n[Invalid coordinate IDs]")
    if invalid_coordinate_ids:
        for gene_id in sorted(invalid_coordinate_ids):
            print(gene_id)
    else:
        print("None")

    print("\n[TSV/FASTA length mismatch IDs]")
    if length_mismatch_ids:
        for gene_id, tsv_len, fasta_len in sorted(length_mismatch_ids):
            print(
                f"{gene_id}\tTSV_length={tsv_len}"
                f"\tFASTA_length={fasta_len}"
            )
    else:
        print("None")

    print(f"\nBad TSV lines: {len(bad_lines)}")
    if bad_lines:
        print("Line numbers:", ", ".join(map(str, bad_lines)))


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

TSV_PATH = "aor0-5_SM00066.tsv"   # TSV without a header
FAA_PATH = "aor0-5_2.faa"         # amino acid sequences of all genes
OUT_FASTA = "aor0-5_SM00066_FL.faa"


def read_fasta_as_dict(faa_file):
    seqs = {}
    cur_id = None
    cur_seq = []

    with open(faa_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    seqs[cur_id] = "".join(cur_seq)
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line.strip())

        if cur_id is not None:
            seqs[cur_id] = "".join(cur_seq)

    return seqs


def main():
    # ---- File existence check ----
    for p in (TSV_PATH, FAA_PATH):
        if not Path(p).exists():
            raise FileNotFoundError(f"File not found: {p}")

    # ---- Load full-length FASTA ----
    seq_dict = read_fasta_as_dict(FAA_PATH)

    # ---- Collect unique IDs from TSV (1st column only) ----
    unique_ids = []
    seen = set()

    with open(TSV_PATH, "r", encoding="utf-8") as tsv:
        for line in tsv:
            line = line.strip()
            if not line:
                continue

            gene_id = line.split("\t")[0]

            if gene_id not in seen:
                seen.add(gene_id)
                unique_ids.append(gene_id)

    # ---- Write full-length sequences ----
    n_written = 0
    n_missing = 0

    with open(OUT_FASTA, "w", encoding="utf-8") as out:
        for gene_id in unique_ids:
            if gene_id not in seq_dict:
                n_missing += 1
                continue

            seq = seq_dict[gene_id]
            header = f">{gene_id}|FL|len={len(seq)}"
            out.write(header + "\n")
            out.write(seq + "\n")
            n_written += 1

    # ---- Report ----
    print("Done.")
    print(f"Unique IDs in TSV: {len(unique_ids)}")
    print(f"Written sequences: {n_written}")
    print(f"Missing IDs in FASTA: {n_missing}")
    print(f"Output file: {OUT_FASTA}")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3

import pandas as pd

# ===== Input files =====
id_chr_file = "CaoGD_aor-05_ID_Chr_AASeq.tsv"
ipr_file = "aor0-5_SM00066.tsv"

# ===== Output files =====
out_file = "aor0-5_SM00066_ProteinID_Chromosome.tsv"
# =======================

# --- ID-to-chromosome mapping table ---
df_id_chr = pd.read_csv(id_chr_file, sep="\t")

id_col = df_id_chr.columns[0]  # the first column contains IDs
id_to_chr = dict(zip(df_id_chr[id_col], df_id_chr["Chromosome"]))

# --- InterProScan TSV (no header) ---
df_ipr = pd.read_csv(ipr_file, sep="\t", header=None)

# First column (Protein ID)
df_ipr["Protein_ID"] = df_ipr[0]

# ID after removing "-P" and everything after it
df_ipr["Base_ID"] = df_ipr["Protein_ID"].str.replace(r"-P.*$", "", regex=True)

# Map chromosomes
df_ipr["Chromosome"] = df_ipr["Base_ID"].map(id_to_chr)

# Output
df_out = df_ipr[["Protein_ID", "Chromosome"]]

df_out.to_csv(out_file, sep="\t", index=False)

print(f"Output written to: {out_file}")


Extract rows from aor0-5_merged.tsv corresponding to SM00066 IDs

In [ ]:
import csv

tsv_file = "aor0-5_merged.tsv"
id_file = "aor0-5_SM00066_ID_list.txt"
output_file = "aor0-5_SM00066_extracted.tsv"

# Read IDs
with open(id_file) as f:
    id_set = set(line.strip() for line in f if line.strip())

with open(tsv_file) as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.reader(infile, delimiter="\t")
    writer = csv.writer(outfile, delimiter="\t")

    header = next(reader)
    writer.writerow(header)

    for row in reader:
        if row[0] in id_set:  # Change this position if the ID column differs
            writer.writerow(row)

print("Done.")

Add a new "non-DBD sequence" column at the right edge of the XLSX file and write the sequence for each matching ID from the supplied FAA file

In [ ]:
from pathlib import Path
from openpyxl import load_workbook

XLSX_PATH = "aor0-5_SM00066_Protein_Seq_list.xlsx"
FAA_PATH = "aor0-5_non_SM00066_domains.faa"
OUT_XLSX = "aor0-5_SM00066_Protein_Seq_list_with_non_DBD.xlsx"

SHEET_NAME = "Sheet1"   # Change as needed


def read_fasta(faa_file):
    """Read FASTA as a {ID: sequence} dictionary"""
    seqs = {}

    cur_id = None
    cur_seq = []

    with open(faa_file, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if cur_id is not None:
                    seqs[cur_id] = "".join(cur_seq)

                # >AO090003001379-P|non_SM00066|...
                fasta_id = line[1:].split("|")[0]

                # Remove a terminal -P
                if fasta_id.endswith("-P"):
                    fasta_id = fasta_id[:-2]

                cur_id = fasta_id
                cur_seq = []

            else:
                cur_seq.append(line)

        if cur_id is not None:
            seqs[cur_id] = "".join(cur_seq)

    return seqs


def main():

    if not Path(XLSX_PATH).exists():
        raise FileNotFoundError(XLSX_PATH)

    if not Path(FAA_PATH).exists():
        raise FileNotFoundError(FAA_PATH)

    fasta = read_fasta(FAA_PATH)

    wb = load_workbook(XLSX_PATH)
    ws = wb[SHEET_NAME]

    # Rightmost column
    new_col = ws.max_column + 1

    ws.cell(row=1, column=new_col).value = "non-DBD sequence"

    matched = 0
    missing = []

    # Rows 2 and below
    for row in range(2, ws.max_row + 1):

        gene_id = ws.cell(row=row, column=3).value   # column C

        if gene_id is None:
            continue

        gene_id = str(gene_id).strip()

        seq = fasta.get(gene_id)

        if seq is None:
            missing.append(gene_id)
        else:
            ws.cell(row=row, column=new_col).value = seq
            matched += 1

    wb.save(OUT_XLSX)

    print(f"Matched IDs : {matched}")
    print(f"Missing IDs : {len(missing)}")

    if missing:
        print("\nMissing IDs")
        for x in missing:
            print(x)

    print(f"\nOutput : {OUT_XLSX}")


if __name__ == "__main__":
    main()